In [ ]:
# ==========================================================
# INSTALL REQUIRED PACKAGES
# (Skip this cell if already installed)
# ==========================================================



!pip install snowflake-snowpark-python pandas numpy scikit-learn matplotlib

In [ ]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from snowflake.snowpark.context import get_active_session

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    ConfusionMatrixDisplay
)

import sklearn

sklearn.set_config(display="text")

# ==========================================================
# CONNECT TO SNOWFLAKE
# ==========================================================

session = get_active_session()

session.sql("USE DATABASE INDUSTRIAL_IOT_QUALITY").collect()
session.sql("USE SCHEMA CURATED").collect()

print("Connected to Snowflake")

In [ ]:
# ==========================================================
# LOAD CURATED ML DATA
# ==========================================================

snowpark_df = session.table(
    "CURATED.ML_ANOMALY_FEATURES"
)

snowpark_df.limit(10).show()

In [ ]:
# ==========================================================
# CONVERT TO PANDAS
# ==========================================================

df = snowpark_df.to_pandas()

df.head()

In [ ]:
# ==========================================================
# BASIC DATA CHECKS
# ==========================================================

print("Rows:", len(df))
print()

print(df.dtypes)

In [ ]:
# ==========================================================
# MISSING VALUES
# ==========================================================

df.isnull().sum()

In [ ]:
# ==========================================================
# DATA TYPES
# ==========================================================

df["EVENT_TS"] = pd.to_datetime(
    df["EVENT_TS"]
)

In [ ]:
# ==========================================================
# SORT BY MACHINE + TIME
# Required before creating rolling features
# ==========================================================

df = (
    df
    .sort_values(
        ["MACHINE_ID", "EVENT_TS"]
    )
    .reset_index(drop=True)
)

df.head()

In [ ]:
# ==========================================================
# CREATE ROLLING FEATURES
# ==========================================================

WINDOW = 10  # was 5

roll_cols = ["TEMPERATURE", "PRESSURE", "VIBRATION", "CYCLE_TIME", "PARTS_PER_HOUR"]  # added PARTS_PER_HOUR

for col in roll_cols:
    grouped = df.groupby("MACHINE_ID")[col]

    df[f"{col}_ROLL_MEAN"] = grouped.transform(
        lambda s: s.shift(1).rolling(window=WINDOW, min_periods=3).mean()
    )
    df[f"{col}_ROLL_STD"] = grouped.transform(
        lambda s: s.shift(1).rolling(window=WINDOW, min_periods=3).std()
    )

    # NEW: short-vs-long trend (captures a ramping degradation, not just level)
    short = grouped.transform(lambda s: s.shift(1).rolling(3, min_periods=2).mean())
    long_ = grouped.transform(lambda s: s.shift(1).rolling(WINDOW, min_periods=3).mean())
    df[f"{col}_TREND"] = short - long_

    # NEW: how far the CURRENT reading is from its recent baseline
    df[f"{col}_DEV_FROM_ROLL"] = df[col] - df[f"{col}_ROLL_MEAN"]

In [ ]:
# ==========================================================
# CREATE FUTURE FAILURE LABEL
# ==========================================================

HORIZON = 15

df["FAILURE_WITHIN_HORIZON"] = (

    df.groupby("MACHINE_ID")["ACTUAL_FAILURE_FLAG"]

      .transform(

          lambda s:

          s[::-1]

          .rolling(
              HORIZON,
              min_periods=1
          )

          .max()

          [::-1]

      )

)

In [ ]:
# ==========================================================
# TARGET DISTRIBUTION
# (Current failure events)
# ==========================================================

print(
    df["FAILURE_WITHIN_HORIZON"].value_counts()
)

print()

print(
    df["FAILURE_WITHIN_HORIZON"]
      .value_counts(normalize=True)
)

In [ ]:
df[
    [
        "MACHINE_ID",
        "EVENT_TS",
        "ACTUAL_FAILURE_FLAG",
        "FAILURE_WITHIN_HORIZON"
    ]
].head(100)

In [ ]:
feature_columns = (
    roll_cols
    + [f"{c}_ROLL_MEAN" for c in roll_cols]
    + [f"{c}_ROLL_STD" for c in roll_cols]
    + [f"{c}_TREND" for c in roll_cols]
    + [f"{c}_DEV_FROM_ROLL" for c in roll_cols]
)

model_df = df.dropna(
    subset=feature_columns
).copy()

X = model_df[feature_columns]

y = model_df["FAILURE_WITHIN_HORIZON"]

In [ ]:
# ==========================================================
# TIME-BASED TRAIN / TEST SPLIT
# ==========================================================

model_df = model_df.sort_values(
    "EVENT_TS"
)

split_index = int(
    len(model_df) * 0.80
)

train = model_df.iloc[:split_index]

test = model_df.iloc[split_index:]

X_train = train[feature_columns]
y_train = train["FAILURE_WITHIN_HORIZON"]

X_test = test[feature_columns]
y_test = test["FAILURE_WITHIN_HORIZON"]

print("Training rows:", len(train))
print("Testing rows :", len(test))

print()
print("Training failure rate:")
print(y_train.mean())

print()
print("Testing failure rate:")
print(y_test.mean())

In [ ]:
# ==========================================================
# RANDOM FOREST MODEL
# ==========================================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(

    n_estimators=300,

    max_depth=12,

    min_samples_leaf=10,

    random_state=42,

    class_weight="balanced",

    n_jobs=-1

)

rf_model.fit(
    X_train,
    y_train
)

In [ ]:
# ==========================================================
# MODEL PREDICTIONS
# ==========================================================

y_pred = rf_model.predict(
    X_test
)

y_prob = rf_model.predict_proba(
    X_test
)[:,1]

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print(classification_report(
    y_test,
    y_pred
))

print(confusion_matrix(
    y_test,
    y_pred
))

print()

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        y_prob
    )
)

In [ ]:
# ==========================================================
# TRAIN VS TEST ROC-AUC
# ==========================================================

train_prob = rf_model.predict_proba(
    X_train
)[:,1]

test_prob = rf_model.predict_proba(
    X_test
)[:,1]

print(
    "Train ROC-AUC:",
    roc_auc_score(
        y_train,
        train_prob
    )
)

print(
    "Test ROC-AUC:",
    roc_auc_score(
        y_test,
        test_prob
    )
)

In [ ]:
# ==========================================================
# FEATURE IMPORTANCE
# ==========================================================

importance = pd.DataFrame({

    "Feature": feature_columns,

    "Importance": rf_model.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

importance

In [ ]:
plt.figure(figsize=(10,6))

plt.barh(

    importance["Feature"],

    importance["Importance"]

)

plt.gca().invert_yaxis()

plt.title(
    "Random Forest Feature Importance"
)

plt.xlabel("Importance")

plt.show()

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(
    y_test,
    y_prob
)

plt.figure(figsize=(6,6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {roc_auc_score(y_test, y_prob):.3f}"
)

plt.plot(
    [0,1],
    [0,1],
    "--"
)

plt.xlabel("False Positive Rate")

plt.ylabel("True Positive Rate")

plt.title("ROC Curve")

plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)

precision, recall, _ = precision_recall_curve(
    y_test,
    y_prob
)

ap = average_precision_score(
    y_test,
    y_prob
)

plt.figure(figsize=(6,6))

plt.plot(
    recall,
    precision,
    label=f"AP = {ap:.3f}"
)

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("Precision-Recall Curve")

plt.legend()

plt.show()

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(

    y_test,

    y_pred

)

plt.show()

In [ ]:
export_df = test.copy()
export_df["PREDICTED_FAILURE_PROBABILITY"] = y_prob
export_df["PREDICTED_RISK_LEVEL"] = pd.cut(
    y_prob, bins=[-0.01, 0.33, 0.66, 1.0], labels=["Low", "Medium", "High"]
)

powerbi_cols = [
    "MACHINE_ID", "EVENT_TS",
    "TEMPERATURE", "PRESSURE", "VIBRATION", "CYCLE_TIME", "PARTS_PER_HOUR",
    "FAILURE_WITHIN_HORIZON", "PREDICTED_FAILURE_PROBABILITY", "PREDICTED_RISK_LEVEL"
]

export_df[powerbi_cols].to_csv("powerbi_predictions.csv", index=False)
print("Saved powerbi_predictions.csv:", len(export_df), "rows")